# 01 · Retrieval as an experimental system

A complete CPU experiment: sentence offsets → lexical ranking → rank fusion → metrics. No downloads. Run from the repository root or notebook directory. This tiny pedagogical corpus is not a benchmark.

In [ ]:
from evidencelab.retrieval import Index
from evidencelab.schema import Paper
from evidencelab.evaluation import ranking_metrics
corpus = [Paper(id="a",title="Adapters",text="Low rank adapters reduce trainable parameters."), Paper(id="b",title="Search",text="BM25 retrieves evidence using lexical term overlap."), Paper(id="c",title="Cells",text="Cell biology studies protein expression.")]
index = Index(corpus)
print(len(index.chunks), "chunks")

## BM25 and TF-IDF

$\mathrm{BM25}(q,d)=\sum_{t\in q}\log(1+\frac{N-df_t+0.5}{df_t+0.5})\frac{tf_{t,d}(k_1+1)}{tf_{t,d}+k_1(1-b+b|d|/\overline{|d|})}$

Here $k_1=1.5$, $b=0.75$. TF-IDF uses $(1+\log tf) [1+\log((N+1)/(df+1))]$ followed by L2 normalization. Compare scores only within a scoring function: BM25 and cosine scores have different scales.

In [ ]:
query = "low rank adapters"
for mode in ["bm25", "tfidf", "hybrid", "graph"]:
    hits = index.search(query, mode=mode, k=2, rewrite=False)
    print(mode, [(h["paper_id"],h["score"]) for h in hits])

## Reciprocal rank fusion

$RRF(d)=\sum_r 1/(60+rank_r(d))$. Rankings, rather than raw scores, are fused. With the neural backend off, both component rankings are lexical. Deduplicate papers after chunk ranking.

In [ ]:
hits=index.search(query,k=3)
metrics=ranking_metrics([h["paper_id"] for h in hits],["a"],3)
assert metrics["recall"] == 1
print(metrics)
for c in index.chunks:
    original=next(p for p in corpus if p.id==c.paper_id)
    assert original.text[c.start:c.end] == c.text

## Failure analysis

Try a paraphrase without shared terms. Lexical retrieval may abstain despite semantic relevance. Then test optional neural retrieval on the exact same queries. Inspect false positives, not just aggregate recall. PCA visualization is a projection of representations, not a measurement of retrieval quality.